# Black-box agents

In this notebook we use [sand-bob](https://haesleinhuepf.github.io/sand-bob/), an agentic AI-system, that analyses data by generating and executing code. For safety reasons, the code is executed in [docker](https://docs.docker.com/engine/install/) containers. You should not execute this notebook if you are not aware what docker is, and what constraints the technology brings.

Can hear more about this topic in [Robert Haase's talk](https://zenodo.org/records/24052892) on Thursday of the Helmholtz Imaging Conference 2026.

For running this notebook, you need to install docker and sand-bob. The latter can be installed like this:

```
pip install sand-bob==0.3.1
```

First, we connect sand-bob to Helmholtz Blablador at FZ Jülich.

In [1]:
from sand_bob import generate_code, config_blablador

config_blablador()

We then define an `agent` function that hides complex procedures. As we cannot see how it performs the analysis, it is considered a black-box agent that presumably generates code and executes it.
Below you find a configuration that does not fix errors if they appear and it does not go through feedback cycles to improve code iteratively.

In [2]:
def agent(prompt, n=1):
    from collections.abc import Iterable
    results = generate_code(prompt, 
                  input_host_path="./input_data/",
                  dependencies=["scikit-image"],
                  n_codefix_attempts=0, # supress error fixing
                  n_feedback_iterations=0, # supress code feedback
                  check_results=False, # do not test resulting objects/types
                  final_touch=False, # do not modify code when done
                  n_parallel=n # number of parallel executions
             )

    if isinstance(results, Iterable):
        return [r.final_result for r in results]
    else:
        return results.final_result

We can then call the agent to analyse a dataset. Per default it does it once, similar to how chat agent systems such as ChatGPT work.

In [3]:
agent("Count the number of bright objects in 'input_data/blobs.tif'. The final result should be a number")

62

Additionally, we can ask the agent to perform the same analysis multiple times. In this case, we can see that not all 10 execution lead to a successful execution.

In [4]:
agent("Count the number of bright objects in 'input_data/blobs.tif'. The final result should be a number", n=10)

[61, None, b'', 64, b'', b'', b'', None, None, None]

## Under the hood

In the following we do this again to see the full output of such a code-generation/+execution. Users can dive into all details, read prompts and see outputs.

In [5]:
results = generate_code("Count the number of bright objects in 'input_data/blobs.tif'. The final result should be a number", 
                  input_host_path="./input_data/",
                  dependencies=["scikit-image"],
                  n_codefix_attempts=0, # supress error fixing
                  n_feedback_iterations=0, # supress code feedback
                  check_results=False, # do not test resulting objects/types
                  final_touch=False, # do not modify code when done
                  n_parallel=3 # number of parallel executions
             )

results

Output()